# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and complies with the Croissant metadata standard, offering rich metadata for AI-ready data infrastructure.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Title:", metadata.name)
print("Dataset Description:", metadata.description)
print("Dataset Keywords:", getattr(metadata, 'keywords', []))
print("Published Date:", getattr(metadata, 'datePublished', 'unknown'))
print("License:", getattr(metadata, 'license', 'unknown'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Each Croissant dataset is composed of `RecordSet` entities. Each `RecordSet` has fields and columns, each with a unique `@id`. These IDs are crucial for referencing entities across metadata and in programmatic exploration.

Let's list all record sets along with their available fields and columns.

In [ ]:
# List all record sets and their fields using `@id`
record_sets = dataset.metadata.recordSet

overview = {}
print("Available Record Sets and their Fields/Columns:")
for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', rs_id)
    print(f"RecordSet @id: {rs_id} ({rs_name})")
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    - {f['@id']} ({f.get('name', f['@id'])})")
    if columns:
        print("  Columns:")
        for c in columns:
            print(f"    - {c['@id']} ({c.get('name', c['@id'])})")
    overview[rs_id] = {
        'fields': [f['@id'] for f in fields] if fields else [],
        'columns': [c['@id'] for c in columns] if columns else []
    }
print("\nFirst record from each RecordSet:")
for rs_id in overview.keys():
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            print(f"RecordSet {rs_id}: Example record:")
            pprint.pprint(recs[0])
    except Exception as e:
        print(f"Could not retrieve records from {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll pick the primary survey record set (usually the one holding participant responses), using its `@id` from above. All references of entities (record sets, fields, columns) will be via their `@id`.

In [ ]:
# Select the main survey RecordSet @id
# For this dataset, the key RecordSet is called 'Kilifi Mental Health Survey Responses'
# We'll retrieve its @id from our overview

# Identify by name or pick the first RecordSet if only one
main_rs_id = None
for rs_id, info in overview.items():
    if 'mental_health' in rs_id.lower() or 'kilifi' in rs_id.lower():
        main_rs_id = rs_id
        break
if not main_rs_id:
    main_rs_id = next(iter(overview))

print(f"Selected primary RecordSet @id: {main_rs_id}")

# Extract all records into a DataFrame
records = list(dataset.records(record_set=main_rs_id))
df = pd.DataFrame(records)
print("Fields (columns) in DataFrame:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and categorizing data.

We'll select numeric fields related to mental health assessments (e.g., GAD-7, PHQ-9, PSQ scores) by searching for their column `@id`. All manipulation references use the actual `@id` keys.

In [ ]:
# Find typical numeric field for analysis (e.g., GAD-7 score)
numeric_field_ids = [col for col in df.columns if any(x in col.lower() for x in ['gad7', 'phq9', 'psq'])]
print("Numeric fields detected:", numeric_field_ids)

# Use first available numeric field (e.g., GAD-7 score field @id)
numeric_field = numeric_field_ids[0] if numeric_field_ids else df.columns[0]
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a common demographic categorical variable, e.g., gender
group_field_ids = [col for col in df.columns if 'gender' in col.lower() or 'sex' in col.lower()]
group_field = group_field_ids[0] if group_field_ids else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}: Mean {numeric_field} per group:")
    print(grouped_df.head())
else:
    print("No suitable categorical group field found for grouping.")

## 5. Visualization
Visualize data distributions and relationships between fields.

We'll chart the distribution of the selected numeric mental health score and compare mean scores across demographic groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# Barplot of mean score per group, if group_field exists
if group_field:
    plt.figure(figsize=(6,4))
    sns.barplot(x=group_field, y=numeric_field, data=df, ci='sd')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(f"{group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and programmatically exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the Croissant metadata schema and the `mlcroissant` library. By referencing all entities by their unique `@id`s, we ensured reproducible and precise metadata-driven workflows. The dataset is suitable for research into mental health patterns and demographic factors, and is ready for further advanced AI/ML analysis and FAIR data integration.